In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import os
import sys

sys.path.append('library')
from atmocean.visualize import (
    get_colormap_from_cmap,
    get_sst_bias_colormap,
    plot_map_on_grid,
)
from atmocean.enso import (
    monthly_resample_with_prediction_timedelta,
    monthly_groupby_with_prediction_timedelta,
    compute_brier_skill_score,
    compute_brier_score,
)
from atmocean.utils import (
    clean_forecasts,
    remove_ocean_variables_suffix,
    align_prediction_timedelta,
    align_prediction_timedelta_optim,
)
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
# get the forecasting horizon for this run
start_date = '01-01'
end_date = '12-31'
lead_times = [0, 10, 20, 30, 40, 50, 60]

main_dir = 'PATH-TO-MAIN-DIRECTORY'
data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/physical_coherency/bss')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/physical_coherency/bss')

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

n_workers = 4
memory_limit = 948 // n_workers
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=1,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
# forecasts = xr.open_mfdataset(forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts)
forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)
# manipulate the time dimension to align
if '_025' not in selected_model:
    forecasts = forecasts.isel(prediction_timedelta=lead_times)
forecasts = forecasts.isel(depth=0, level=0, missing_dims='ignore')
forecasts = remove_ocean_variables_suffix(forecasts)

In [ ]:
# select only meridional and zonal velocities
forecasts = forecasts['siconc']

def align_prediction_timedelta_old(ds: xr.Dataset):
    lead_days = (
        ds['prediction_timedelta']
        .values
        .astype('timedelta64[D]')
        .astype(int)
    )
    lead_days = lead_days - 1
    dss = []
    for i, days in enumerate(lead_days):
        ds_lead = ds.isel(prediction_timedelta=i).copy()
        ds_lead['time'] = (
            pd.to_datetime(ds_lead['time'].values) + pd.DateOffset(days=int(days))
        )
        dss.append(ds_lead)
    return xr.concat(dss, dim='prediction_timedelta')

# groupby the climatology
forecasts = align_prediction_timedelta_old(forecasts)

In [ ]:
forecasts_brier_score = compute_brier_score(forecasts)

In [ ]:
forecasts_brier_score = client.persist(forecasts_brier_score)
progress(forecasts_brier_score)
wait(forecasts_brier_score)

In [ ]:
if ensemble_size is None:
    print('no ensemble')
    forecasts_brier_score = forecasts_brier_score.load()
else:
    print('ensemble')
    forecasts_brier_score = forecasts_brier_score.mean(dim='ensemble_member', skipna=True).load()

In [ ]:
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:
grid_list = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[2,2:4]',
]

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(0, 0.3)

projection = ccrs.NorthPolarStereo()
extent = [0, 360, 60, 90]

brier_score_list = []
for i,lt in enumerate(lead_times):
    brier_score_list.append(
        forecasts_brier_score.isel(prediction_timedelta=i)
    )

lats, lons = brier_score_list[0].latitude.data, brier_score_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(10,5.5))
fig.suptitle(f'Brier Score over North Pole', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times):
    im, ax = plot_map_on_grid(
        bias=brier_score_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, projection=projection, 
        cmap=cmap, norm=norm, levels=levels, draw_labels=False
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.03, top=0.95, wspace=-0.8, hspace=0.2
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_spatial_north_pole.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(0, 0.2)

projection = ccrs.NorthPolarStereo()
extent = [0, 360, 60, 90]

brier_score_list = []
for i,lt in enumerate(lead_times[1:]):
    brier_score_list.append(
        forecasts_brier_score.isel(prediction_timedelta=i+1) - forecasts_brier_score.isel(prediction_timedelta=0)
    )

lats, lons = brier_score_list[0].latitude.data, brier_score_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(15,5.5))
fig.suptitle(f'Brier Score Bias over North Pole', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=brier_score_list[i], grid=eval(grid_list[i]), title=f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, projection=projection, 
        cmap=cmap, norm=norm, levels=levels, draw_labels=False
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.03, top=0.95, wspace=-0.8, hspace=0.2
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_bias_spatial_north_pole.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(0, 0.3)

projection = ccrs.SouthPolarStereo()
extent = [0, 360, -90, -50]

brier_score_list = []
for i,lt in enumerate(lead_times):
    brier_score_list.append(
        forecasts_brier_score.isel(prediction_timedelta=i)
    )

lats, lons = brier_score_list[0].latitude.data, brier_score_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(10,5.5))
fig.suptitle(f'Brier Score over South Pole', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times):
    im, ax = plot_map_on_grid(
        bias=brier_score_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, projection=projection, 
        cmap=cmap, norm=norm, levels=levels, draw_labels=False
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.03, top=0.95, wspace=-0.8, hspace=0.2
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_spatial_south_pole.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(0.0, 0.2)

projection = ccrs.SouthPolarStereo()
extent = [0, 360, -90, -50]

brier_score_list = []
for i,lt in enumerate(lead_times[1:]):
    brier_score_list.append(
        forecasts_brier_score.isel(prediction_timedelta=i+1) - forecasts_brier_score.isel(prediction_timedelta=0)
    )

lats, lons = brier_score_list[0].latitude.data, brier_score_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(15,5.5))
fig.suptitle(f'Brier Score Bias over South Pole', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []

for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=brier_score_list[i], grid=eval(grid_list[i]), title=f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, projection=projection, 
        cmap=cmap, norm=norm, levels=levels, draw_labels=False
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.03, top=0.95, wspace=-0.8, hspace=0.2
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_bias_spatial_south_pole.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0]',
    'gs[0,1]',
    'gs[0,2]',
    'gs[0,3]',
]
grid_list_bottom = [
    'gs[1,0]',
    'gs[1,1]',
    'gs[1,2]',
    'gs[1,3]',
]

projection = ccrs.NorthPolarStereo()
extent = [0, 360, 60, 90]

cmap_top, norm_top, levels_top = get_sst_bias_colormap(0, 0.3)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', 0.0, 0.2)

bs_list = []
bs_bias_list = []
for i,lt in enumerate(lead_times_figure):
    bs_list.append(
        forecasts_brier_score.isel(prediction_timedelta=lt//10)
    )
    bs_bias_list.append(
        forecasts_brier_score.isel(prediction_timedelta=lt//10) - forecasts_brier_score.isel(prediction_timedelta=0)
    )

lats, lons = bs_list[0].latitude.data, bs_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(6,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 4, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=bs_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=None, 
        cmap=cmap_top, norm=norm_top, levels=levels_top, 
        projection=projection, draw_labels=False, 
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=bs_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, 
        cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom, 
        projection=projection, draw_labels=False, 
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_north.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0]',
    'gs[0,1]',
    'gs[0,2]',
    'gs[0,3]',
]
grid_list_bottom = [
    'gs[1,0]',
    'gs[1,1]',
    'gs[1,2]',
    'gs[1,3]',
]

projection = ccrs.SouthPolarStereo()
extent = [0, 360, -90, -50]

cmap_top, norm_top, levels_top = get_sst_bias_colormap(0, 0.3)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', 0.0, 0.2)

bs_list = []
bs_bias_list = []
for i,lt in enumerate(lead_times_figure):
    bs_list.append(
        forecasts_brier_score.isel(prediction_timedelta=lt//10)
    )
    bs_bias_list.append(
        forecasts_brier_score.isel(prediction_timedelta=lt//10) - forecasts_brier_score.isel(prediction_timedelta=0)
    )

lats, lons = bs_list[0].latitude.data, bs_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(6,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 4, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=bs_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=None, 
        cmap=cmap_top, norm=norm_top, levels=levels_top, 
        projection=projection, draw_labels=False, 
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=bs_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=None, 
        cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom, 
        projection=projection, draw_labels=False, 
    )
    ax.set_extent(extent, ccrs.PlateCarree())
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'brier_score_south.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()